In [9]:
pip install mlxtend --upgrade

Note: you may need to restart the kernel to use updated packages.


In [10]:
import ast
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [11]:
basket = pd.read_csv('customer_basket.csv')
clusters = pd.read_csv('cluster_assignments.csv')
basket = basket.merge(clusters, on='customer_id')

print(f"Total baskets: {len(basket)}")
print(f"Baskets per cluster:\n{basket['cluster'].value_counts().sort_index()}")

Total baskets: 100000
Baskets per cluster:
cluster
0     6450
1    38865
2    30946
3     9768
4    13971
Name: count, dtype: int64


In [12]:
basket['list_of_goods'] = basket['list_of_goods'].apply(ast.literal_eval)

# Quick sanity check
print(f"Sample basket: {basket['list_of_goods'].iloc[0]}")
print(f"Sample cluster: {basket['cluster'].iloc[0]}")

Sample basket: ['chicken', 'rice', 'pepper', 'whole wheat rice', 'shrimp', 'toothpaste', 'gums', 'cereals', 'bluetooth headphones', 'vacuum cleaner', 'body spray']
Sample cluster: 2


In [14]:
def get_association_rules(transactions, min_support=0.02, min_confidence=0.2):
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    df = pd.DataFrame(te_array, columns=te.columns_)
    frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        print("No frequent itemsets found")
        return pd.DataFrame()
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    return rules.sort_values('lift', ascending=False)

In [15]:
all_products = [item for sublist in basket['list_of_goods'] for item in sublist]
print(f"Unique products: {len(set(all_products))}")
print(f"Most common products:\n{pd.Series(all_products).value_counts().head(10)}")

Unique products: 164
Most common products:
asparagus       12811
airpods         12145
cereals          9952
fresh bread      9934
butter           9654
eggs             9241
protein bar      8695
cooking oil      8623
toilet paper     8395
babies food      8318
Name: count, dtype: int64


In [16]:
print(basket['cluster'].value_counts().sort_index())
print(basket['cluster'].isna().sum())

cluster
0     6450
1    38865
2    30946
3     9768
4    13971
Name: count, dtype: int64
0


In [17]:
print(basket[basket['cluster'].isna()].shape)
basket = basket.dropna(subset=['cluster'])
basket['cluster'] = basket['cluster'].astype(int)
print(basket['cluster'].value_counts().sort_index())

(0, 4)
cluster
0     6450
1    38865
2    30946
3     9768
4    13971
Name: count, dtype: int64


In [18]:
# 4. Run per cluster and store results
cluster_rules = {}
for cluster_id in sorted(basket['cluster'].unique()):
    transactions = basket[basket['cluster'] == cluster_id]['list_of_goods'].tolist()
    rules = get_association_rules(transactions)
    cluster_rules[cluster_id] = rules

In [19]:
for cluster_id in sorted(basket['cluster'].unique()):
    print(f"\n{'='*50}")
    print(f"CLUSTER {cluster_id} — Top Unique Rules by Lift")
    print(f"{'='*50}")
    rules = cluster_rules[cluster_id]
    if not rules.empty:
        # Convert frozensets to sorted tuples to identify duplicates
        rules = rules.copy()
        rules['pair'] = rules.apply(
            lambda row: tuple(sorted([
                str(sorted(row['antecedents'])), 
                str(sorted(row['consequents']))
            ])), axis=1
        )
        # Keep only one direction per pair
        unique_rules = rules.drop_duplicates(subset='pair')
        top10 = unique_rules.nlargest(10, 'lift')[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
        display(top10)


CLUSTER 0 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
0,(airpods),(asparagus),0.022481,0.17407,1.073374



CLUSTER 1 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
2,(airpods),(energy drink),0.023878,0.201214,2.646427
0,(airpods),(bluetooth headphones),0.022617,0.190590,2.563070
13,(napkins),(dog food),0.021922,0.250220,2.476397
5,(babies food),(cooking oil),0.021176,0.213434,2.442608
9,(babies food),(napkins),0.021176,0.213434,2.436152
6,(dog food),(babies food),0.023492,0.232493,2.343320
11,(dog food),(cooking oil),0.020018,0.198116,2.267304



CLUSTER 2 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
27,(shampoo),(shower gel),0.025755,0.291088,3.514640
20,(shower gel),(deodorant),0.025658,0.309793,3.500132
32,(shower gel),(tooth brush),0.025011,0.301990,3.355612
19,(deodorant),(shampoo),0.025367,0.286601,3.239282
28,(tooth brush),(shampoo),0.025334,0.281508,3.181720
35,(shower gel),(toothpaste),0.025270,0.305111,3.105912
22,(tooth brush),(deodorant),0.024365,0.270736,3.058853
36,(toothpaste),(tooth brush),0.026401,0.268750,2.986261
31,(shampoo),(toothpaste),0.025948,0.293280,2.985472
24,(toothpaste),(deodorant),0.024979,0.254276,2.872886



CLUSTER 3 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
547,(cereals),"(green tea, eggs)",0.020680,0.059377,1.374391
544,"(eggs, cereals)",(green tea),0.020680,0.145011,1.352880
543,"(green tea, cereals)",(eggs),0.020680,0.458050,1.351732
576,(cereals),"(eggs, salt)",0.025901,0.074368,1.345235
634,"(oatmeal, cereals)",(honey),0.022011,0.270440,1.345041
729,"(fresh bread, honey)",(tea),0.021499,0.269923,1.339059
562,"(oatmeal, cereals)",(eggs),0.036855,0.452830,1.336328
442,"(honey, butter)",(eggs),0.035217,0.452037,1.333987
568,"(cereals, oil)",(eggs),0.021192,0.451965,1.333775
727,(oatmeal),"(fresh bread, honey)",0.021601,0.106190,1.333247



CLUSTER 4 — Top Unique Rules by Lift


,antecedents,consequents,support,confidence,lift
61,(bluetooth headphones),(energy drink),0.026627,0.290625,3.154873
54,(avocado),(salad),0.027915,0.290395,2.931433
62,(salad),(carrots),0.031064,0.313584,2.835650
0,(airpods),(bluetooth headphones),0.043018,0.256290,2.797365
52,(avocado),(carrots),0.028845,0.300074,2.713489
68,(salad),(spinach),0.029275,0.295520,2.684469
6,(airpods),(energy drink),0.041300,0.246055,2.671049
56,(avocado),(spinach),0.028201,0.293373,2.664964
43,(salad),(asparagus),0.046811,0.472543,2.588982
19,(avocado),(asparagus),0.044306,0.460908,2.525236


**Support** - proportion of baskets that contain both items

**Confidence** - percentage of bought together ('If someone buys item A, we have a x% confidence that buys item B)

**Lift** - measure of how much more likely someone is to buy two items together compared to independently
- Lift = 1 → no association, purely coincidental
- Lift > 1 → positive association, bought together more than expected
- Lift < 1 → negative association, rarely bought together

# Cluster Campaigns


**Cluster 0 - "Biggest Buyers"**
- Only 1 meaningful rule (asparagus → airpods, lift 1.07) which is very weak
- Don't base the campaign on association rules here - use their spending profile instead
- Campaign: **"VIP Exclusive — spend over €150 and get 15% off your entire basket"**

**Cluster 1 - "Most Loyal"**
- airpods → energy drink (lift 2.65), airpods → bluetooth headphones (lift 2.56)
- high fresh_food_ratio, hygiene and tech spending 
- has_loyalty_card is 1
- Campaign: **"Tech Bundle - buy AirPods and get a free Energy Drink"** or **"Loyal Member Reward - use your loyalty card and get 10% off everything from the categories 'Fresh Food', 'Hygiene' and 'Tech'"**
maybe change category each week

**Cluster 2 - "New Customers + Tech Enthusiast"**
- Surprisingly dominated by hygiene products - shampoo ↔ shower gel (lift 3.51), deodorant ↔ shower gel (lift 3.50), shower gel → toothbrush (lift 3.36), toothpaste → deodorant (lift 2.87)
- high fresh_food_ratio and tech spending 
- Campaign: **"Welcome Hygiene Kit - buy shampoo + shower gel + deodorant and get a toothbrush free"** and **"Fresh Tech Deal - buy €50 in fresh food and get 10% off any electronics"**
maybe add incentive to create loyalty card

**Cluster 3 - "Big Family Shopper"**
- eggs + green tea → cereals (lift 1.37), cereals + oatmeal → honey (lift 1.35), honey + fresh bread → tea (lift 1.34) - Strong healthy breakfast pattern
- high drinks (alcohol and non-alcohol) and hygiene spending
- Campaign: **"Family Breakfast Bundle - buy cereals + oatmeal and get honey 30% off"**, **"Party Ready Bundle - buy 3 alcoholic drinks and get 20% off non-alcoholic drinks"** and **"Family Hygiene Pack - buy €30 in hygiene products and get the cheapest one free"**

**Cluster 4 - "Bargain Hunter"**
- bluetooth headphones → energy drink (lift 3.15), avocado → salad (lift 2.93), salad → carrots (lift 2.84), airpods → bluetooth headphones (lift 2.80), avocado → spinach (lift 2.66)
- spend mostly on sales
- Campaign: **"Healthy Plate Deal - buy avocado + salad and get carrots + spinach at half price"** and **"Double Promotion Day — every Wednesday all your usual promoted items get an extra 10% off"**